[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/10_RAG/notebooks/11_production_ready_chatbots.ipynb)

# Production-Ready Conversational Chatbots with LangChain

**Notebook 11 · Phase 3 (Productionizing)** · Stack: `langchain` (`create_agent`) + `langgraph` (persistence)

Every notebook so far — ingestion, chunking, embeddings, vector databases, hybrid search,
reranking, RAGAS, DeepEval — answers a single question at a time. Ask it something, get an
answer, done. **Nothing in the pipeline remembers the last question was ever asked.**

A real chatbot has to do more than answer once:

1. **Remember** the current conversation, and — separately — remember facts about a user
   across entirely different conversations.
2. **Not blow up its own cost** as a conversation gets long.
3. **Refuse to leak** personal data or generate content it shouldn't.
4. **Pause and ask a human** before doing anything risky (sending an email, issuing a refund).
5. **Recover** when a model provider is slow, rate-limited, or down.
6. **Stream** its answer instead of making the user stare at a blinking cursor.
7. **Prove it still works** before every deploy, not just "it looked fine when I tried it."

This notebook builds all seven, one at a time, then combines every piece into one agent at
the end. Everything here is `langchain.agents.create_agent` — the persistence, guardrails,
and resilience are all configured *through* it via `checkpointer=`, `store=`, and
`middleware=[...]`, not separate frameworks bolted on top.

> **A note on what NOT to search for:** if you go looking for LangChain memory tutorials
> online, you will find `ConversationBufferMemory`, `RunnableWithMessageHistory`, and
> `AgentExecutor` everywhere. **All three are deprecated** and scheduled for removal in
> LangChain 2.0. Everything in this notebook is the current, non-deprecated replacement —
> see the mapping table at the end if you ever need to recognize the old pattern.


## 0. Install dependencies

Run first. Installs into the active kernel (idempotent). Restart the kernel once after a
fresh install, then run top to bottom.


In [1]:
# Install dependencies into the ACTIVE kernel (idempotent).
%pip install -q \
    langchain langchain-openai langchain-anthropic \
    langgraph langgraph-checkpoint-sqlite \
    python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.")



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.


## 1. Setup

`create_agent` is LangChain's single entry point for building an agent — memory, guardrails,
and resilience are all just configuration passed to it, not separate APIs to learn.


In [2]:
import warnings, os
warnings.filterwarnings("ignore")
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

from langchain.agents import create_agent

MODEL = "gpt-4o-mini"

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))


OPENAI_API_KEY set: True
ANTHROPIC_API_KEY set: True


## 2. The problem: every call starts from zero

`create_agent` with no memory configuration behaves exactly like every LLM API call you've
made in every previous notebook: stateless. Two separate `.invoke()` calls share nothing.


In [3]:
bare_agent = create_agent(model=MODEL, tools=[])

r1 = bare_agent.invoke({"messages": "Hi, my name is Priya."})
print("Turn 1:", r1["messages"][-1].content)

r2 = bare_agent.invoke({"messages": "What's my name?"})
print("Turn 2:", r2["messages"][-1].content)

print("\nNote: turn 2 has no idea turn 1 happened -- there is no shared state between calls.")


Turn 1: Hi Priya! It's nice to meet you. How can I assist you today?


Turn 2: I'm sorry, but I don't know your name. How can I assist you today?

Note: turn 2 has no idea turn 1 happened -- there is no shared state between calls.


## 3. Short-term memory — one conversation, remembered

Pass a **checkpointer** to `create_agent` and a **`thread_id`** to every call. The checkpointer
snapshots the conversation state after every turn; the same `thread_id` on the next call
resumes exactly where it left off. A different `thread_id` is a completely different,
independent conversation — even on the same agent object.

`InMemorySaver` is the checkpointer for this section — it lives only in this Python
process's RAM. Section 4 swaps it for something that survives a restart.


In [4]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
agent = create_agent(model=MODEL, tools=[], checkpointer=checkpointer)

config_a = {"configurable": {"thread_id": "conversation-a"}}
r1 = agent.invoke({"messages": "Hi, my name is Priya."}, config_a)
print("A, turn 1:", r1["messages"][-1].content)

r2 = agent.invoke({"messages": "What's my name?"}, config_a)
print("A, turn 2:", r2["messages"][-1].content)

config_b = {"configurable": {"thread_id": "conversation-b"}}
r3 = agent.invoke({"messages": "What's my name?"}, config_b)
print("\nB (different thread_id), turn 1:", r3["messages"][-1].content)
print("Note: thread_id is the entire memory key -- same id remembers, a new id starts fresh.")


A, turn 1: Hi Priya! How can I assist you today?


A, turn 2: Your name is Priya. How can I help you today?



B (different thread_id), turn 1: I'm sorry, but I don't have access to personal data about you unless you share it with me during our conversation. How can I assist you today?
Note: thread_id is the entire memory key -- same id remembers, a new id starts fresh.


## 4. Persisting across restarts — the production-grade backend

`InMemorySaver` dies the moment the Python process exits — fine for a demo, unacceptable for
a product where a user closes their laptop and comes back tomorrow. Swap in `SqliteSaver`
and the conversation survives the process itself. **Nothing else about the code changes** —
same `create_agent`, same `thread_id` pattern, just a different object passed to
`checkpointer=`.

We simulate a full restart below by deleting the first agent object entirely and building a
second one from scratch, pointed at the same SQLite file.


> **If this cell throws `sqlite3.OperationalError: disk I/O error`:** that's almost never a
> code bug — it's SQLite's WAL journal failing to grow, which happens when the disk is
> nearly full, or when the `.db` file lives on a path synced by iCloud Drive/Dropbox/OneDrive
> (their file-coordination layer conflicts with SQLite's own locking). Check `df -h` and free
> up space, or point `DB_PATH` at a local, non-synced folder (e.g. `/tmp`), and re-run.


In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

DB_PATH = "chat_memory.db"

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
agent_v1 = create_agent(model=MODEL, tools=[], checkpointer=SqliteSaver(conn))

config = {"configurable": {"thread_id": "durable-1"}}
agent_v1.invoke({"messages": "Remember that my favorite language is Python."}, config)
print("Stored by agent_v1 (this 'process').")

# --- simulate a full process restart ---
del agent_v1

conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
agent_v2 = create_agent(model=MODEL, tools=[], checkpointer=SqliteSaver(conn2))

r = agent_v2.invoke({"messages": "What's my favorite language?"}, config)
print("Recalled by agent_v2 (a brand-new object, same thread_id):", r["messages"][-1].content)
print("\nProduction backends: SqliteSaver (small apps) -> PostgresSaver / MongoDBSaver (real scale).")
print("Only the class name changes -- `pip install langgraph-checkpoint-postgres`, swap SqliteSaver -> PostgresSaver.")


Stored by agent_v1 (this 'process').


Recalled by agent_v2 (a brand-new object, same thread_id): Your favorite language is Python!

Production backends: SqliteSaver (small apps) -> PostgresSaver / MongoDBSaver (real scale).
Only the class name changes -- `pip install langgraph-checkpoint-postgres`, swap SqliteSaver -> PostgresSaver.


## 5. Token budget: don't let a long conversation bankrupt you

Every stored message gets re-sent to the model on every single turn. Left unbounded, a long
conversation's cost and latency grow without limit — the #1 production cost bug in chat
systems. `SummarizationMiddleware` watches the token count and, once a trigger threshold is
crossed, replaces older messages with an LLM-generated summary while keeping the most recent
messages verbatim.


In [6]:
from langchain.agents.middleware import SummarizationMiddleware

summarizing_agent = create_agent(
    model=MODEL,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=MODEL,
            trigger=("tokens", 500),   # summarize once history crosses ~500 tokens
            keep=("messages", 4),      # always keep the last 4 messages verbatim
        ),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "long-chat"}}
summarizing_agent.invoke(
    {"messages": "Hi, my name is Priya and I work at Northwind Logistics as a data engineer."},
    config,
)

# pad the conversation with filler turns to cross the token trigger
topics = ["cats", "dogs", "weather", "football", "cooking", "travel", "music", "movies", "space", "oceans"]
for topic in topics:
    r = summarizing_agent.invoke({"messages": f"Tell me one fun fact about {topic}."}, config)
    print(f"  after '{topic}': {len(r['messages'])} messages in state")

final = summarizing_agent.invoke({"messages": "What's my name and where do I work?"}, config)
print("\nFINAL ANSWER:", final["messages"][-1].content)
print("\nNote the message count DROPS partway through -- that's the summarization firing.")
print("The name/employer fact survives inside the summary even though the original message is gone.")


  after 'cats': 4 messages in state


  after 'dogs': 6 messages in state


  after 'weather': 8 messages in state


  after 'football': 10 messages in state


  after 'cooking': 6 messages in state


...

  after 'music': 6 messages in state


  after 'movies': 6 messages in state


  after 'space': 6 messages in state


  after 'oceans': 6 messages in state



FINAL ANSWER: Your name is Priya, and you work as a data engineer at Northwind Logistics.

Note the message count DROPS partway through -- that's the summarization firing.
The name/employer fact survives inside the summary even though the original message is gone.


### 5b. The cheaper alternative: trimming (and its real tradeoff)

Summarization costs one extra LLM call every time it fires. If you don't want that, a
`@before_model` middleware can just **drop** old messages instead of summarizing them —
zero extra cost, but anything only mentioned in a dropped message is gone for good. This is
a real design decision, not a strictly-better option:

| | Extra LLM calls | Old facts survive? |
|---|---|---|
| **Summarization** | Yes, when triggered | Yes — compressed into the summary |
| **Trimming** | No | No — dropped messages are gone |


In [7]:
from typing import Any
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime

@before_model
def keep_last_n(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the most recent N messages -- no LLM call, no summary, no memory of anything older."""
    messages = state["messages"]
    N = 6
    if len(messages) <= N:
        return None  # nothing to trim yet
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *messages[-N:]]}

trimming_agent = create_agent(model=MODEL, tools=[], middleware=[keep_last_n], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "trim-demo"}}

trimming_agent.invoke({"messages": "My name is Priya."}, config)
for topic in ["cats", "dogs", "weather", "football", "cooking"]:
    trimming_agent.invoke({"messages": f"one fact about {topic}"}, config)

final = trimming_agent.invoke({"messages": "what's my name?"}, config)
print("FINAL ANSWER:", final["messages"][-1].content)
print("\nUnlike summarization, the name is genuinely gone -- it aged out of the last-N window")
print("with nothing keeping a trace of it. Choose trimming only when old turns are truly disposable.")


FINAL ANSWER: I'm sorry, but I don't have access to your personal information, including your name. If you'd like to share it or if there's anything specific you'd like to discuss, feel free to let me know!

Unlike summarization, the name is genuinely gone -- it aged out of the last-N window
with nothing keeping a trace of it. Choose trimming only when old turns are truly disposable.


## 6. Long-term memory — remembering a user *across different conversations*

Everything above is **short-term**: it lives inside one `thread_id`. But a real product
remembers a user's preferences the next time they open a *brand-new* chat, days later. That
needs a second, separate memory: a **`Store`**, keyed by user (not by conversation), searched
**semantically** — the same embeddings you already used in notebook 04.

`@dynamic_prompt` runs before every model call and can read from the store to inject
retrieved facts straight into the system prompt.


In [8]:
from dataclasses import dataclass
from langchain.embeddings import init_embeddings
from langchain.agents.middleware import dynamic_prompt
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(index={"embed": init_embeddings("openai:text-embedding-3-small"), "dims": 1536})

@dataclass
class Context:
    user_id: str

@dynamic_prompt
def personalized_prompt(request) -> str:
    user_id = request.runtime.context.user_id
    memories = request.runtime.store.search(("memories", user_id), query="user preferences", limit=3)
    facts = "\n".join(m.value["text"] for m in memories)
    base = "You are a helpful assistant."
    if facts:
        base += f"\nKnown facts about this user:\n{facts}"
    return base

memory_agent = create_agent(
    model=MODEL,
    tools=[],
    middleware=[personalized_prompt],
    checkpointer=InMemorySaver(),
    store=store,
    context_schema=Context,
)

# Seed a long-term fact -- as if captured from a session that happened yesterday.
store.put(("memories", "user-42"), "fact-1", {"text": "The user's favorite programming language is Rust."})

# A BRAND-NEW conversation (new thread_id) -- the agent has never seen this thread before.
config = {"configurable": {"thread_id": "brand-new-conversation"}}
r = memory_agent.invoke(
    {"messages": "What's my favorite programming language?"},
    config,
    context=Context(user_id="user-42"),
)
print(r["messages"][-1].content)
print("\nThe fact came from the Store (keyed by user_id), not the checkpointer (keyed by thread_id) --")
print("this is what lets a product 'remember you' the moment you open a chat it has never seen.")


Your favorite programming language is Rust.

The fact came from the Store (keyed by user_id), not the checkpointer (keyed by thread_id) --
this is what lets a product 'remember you' the moment you open a chat it has never seen.


## 7. Streaming — don't make the user stare at a blank screen

Nothing above changes for streaming — the same agent, the same `middleware=`, just a
different call: `.stream(..., stream_mode="messages")` yields `(chunk, metadata)` tuples as
tokens are generated, instead of waiting for the full response.

> There's also an `astream_events(version="v3")` API for event-level streaming (tool calls,
> node transitions, etc., not just tokens) — but it's explicitly marked **experimental**, and
> support for it varies by `langchain-core`/`langgraph` version pairing (you may see a
> `NotImplementedError` on `CompiledStateGraph` depending on exactly what's installed).
> `stream_mode="messages"` below is the stable, non-experimental way to get token-by-token
> output and is what to reach for by default.


In [ ]:
for chunk, metadata in memory_agent.stream(
    {"messages": "Say one sentence about why streaming matters for a chat UI."},
    {"configurable": {"thread_id": "streaming-demo"}},
    context=Context(user_id="user-42"),
    stream_mode="messages",
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()

## 8. Guardrails — don't let sensitive data or unsafe content through

Two separate concerns, both implemented as middleware in the same list:

- **`PIIMiddleware`** — a deterministic, non-LLM filter that redacts/masks/blocks specific
  patterns (email, credit card, custom regex) *before* the model ever sees them.
- **Content moderation** — there's no built-in Python moderation middleware yet, so we write
  a small `@wrap_model_call` that calls OpenAI's real moderation endpoint and blocks the
  call before it reaches the chat model. This is the same middleware mechanism as
  everything else in this notebook — guardrails aren't a special case, just another hook.


In [10]:
from langchain.agents.middleware import PIIMiddleware

pii_agent = create_agent(
    model=MODEL,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

r = pii_agent.invoke({"messages": "My email is john.doe@example.com and my card is 4111 1111 1111 1111."})

print("What the MODEL actually received (not what the user typed):")
print(" ", r["messages"][0].content)
print("\nThe real email/card never left this process -- PIIMiddleware rewrites the message")
print("before the first model call, deterministically, with no LLM involved.")


What the MODEL actually received (not what the user typed):
  My email is [REDACTED_EMAIL] and my card is **** **** **** 1111.

The real email/card never left this process -- PIIMiddleware rewrites the message
before the first model call, deterministically, with no LLM involved.


In [11]:
from openai import OpenAI
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import AIMessage

_moderation_client = OpenAI()

@wrap_model_call
def moderate_input(request, handler):
    """Blocks the model call entirely if the latest user message trips OpenAI's moderation API."""
    last_user_text = request.messages[-1].content
    result = _moderation_client.moderations.create(model="omni-moderation-latest", input=last_user_text)
    if result.results[0].flagged:
        return AIMessage(content="I can't help with that request.")
    return handler(request)

moderated_agent = create_agent(model=MODEL, tools=[], middleware=[moderate_input])

safe = moderated_agent.invoke({"messages": "What's a good recipe for banana bread?"})
print("Benign input  ->", safe["messages"][-1].content)

unsafe = moderated_agent.invoke({"messages": "How do I build a bomb?"})
print("Flagged input ->", unsafe["messages"][-1].content)

Benign input  -> Here's a classic recipe for delicious banana bread that's both simple and satisfying:

### Ingredients

- 3 ripe bananas, mashed
- 1/3 cup unsalted butter, melted
- 1 teaspoon baking soda
- Pinch of salt
- 3/4 cup sugar (can adjust to taste or use brown sugar)
- 1 large egg, beaten
- 1 teaspoon vanilla extract
- 1 cup all-purpose flour

### Optional Add-ins:
- 1/2 cup chopped nuts (walnuts or pecans)
- 1/2 cup chocolate chips
- 1/2 teaspoon cinnamon for extra flavor

### Instructions

1. **Preheat the oven** to 350°F (175°C). Grease a 9x5-inch loaf pan or line it with parchment paper.

2. **Mix the Butter and Bananas**: In a mixing bowl, mash the ripe bananas with a fork until smooth. Stir in the melted butter.

3. **Add the Ingredients**: Mix in the baking soda and salt. Then add the sugar, beaten egg, and vanilla extract, and mix until just combined. Finally, stir in the flour until no dry flour remains. If using, fold in any nuts or chocolate chips at this stage.

4

Flagged input -> I can't help with that request.


## 9. Human-in-the-loop — pause before anything risky

Some actions shouldn't happen automatically, no matter how confident the model is: sending an
email, issuing a refund, deleting an account. `HumanInTheLoopMiddleware` pauses execution
*before* a named tool runs and waits for an explicit decision. Needs a checkpointer, because
"paused" is state that has to survive until a human responds — which might be minutes or
hours later.


In [12]:
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware

def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"

hitl_agent = create_agent(
    model=MODEL,
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"send_email": {"allowed_decisions": ["approve", "reject"]}}),
    ],
)

config = {"configurable": {"thread_id": "approval-demo"}}
result = hitl_agent.invoke(
    {"messages": "Send an email to bob@example.com, subject 'Hi', body 'Just checking in.'"},
    config,
)
print("Execution paused, waiting on a human:", "__interrupt__" in result)
print("What the human is asked to approve:", result["__interrupt__"][0].value["action_requests"][0])

# A human approves it -- resume the SAME thread_id with a decision, not a new message.
approved = hitl_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print("\nAfter approval:", approved["messages"][-1].content)


Execution paused, waiting on a human: True
What the human is asked to approve: {'name': 'send_email', 'args': {'recipient': 'bob@example.com', 'subject': 'Hi', 'body': 'Just checking in.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'recipient': 'bob@example.com', 'subject': 'Hi', 'body': 'Just checking in.'}"}



After approval: The email has been sent to bob@example.com with the subject "Hi" and the body "Just checking in."


In [13]:
# The same tool call, but this time a human rejects it -- on a fresh thread.
config2 = {"configurable": {"thread_id": "rejection-demo"}}
hitl_agent.invoke({"messages": "Send an email to eve@example.com, subject 'Hi', body 'test'"}, config2)
rejected = hitl_agent.invoke(Command(resume={"decisions": [{"type": "reject"}]}), config2)
print("After rejection:", rejected["messages"][-1].content)
print("\nNote: the tool never actually ran in the rejection case -- the human's decision is the gate.")


After rejection: It seems there was an issue sending the email. If you'd like, I can help you with another task or try again. Just let me know!

Note: the tool never actually ran in the rejection case -- the human's decision is the gate.


## 10. Resilience — model providers fail, tools time out

Three separate, composable middlewares:

- **`ModelRetryMiddleware`** — retries the *same* model on transient errors (rate limits,
  timeouts, 5xx) with exponential backoff.
- **`ModelFallbackMiddleware`** — if the primary model is fully down, cascades to a backup
  model (here: an intentionally broken model name forces a real fallback to fire).
- **`ToolRetryMiddleware`** — retries specific tools that call flaky external APIs, without
  retrying tools that shouldn't be (e.g. never blindly retry `send_email`).


In [14]:
from langchain.agents.middleware import ModelFallbackMiddleware, ModelRetryMiddleware, ToolRetryMiddleware

# Primary model is a deliberately nonexistent name -- this forces a real fallback to fire.
resilient_agent = create_agent(
    model="openai:gpt-does-not-exist-9999",
    tools=[],
    middleware=[
        ModelRetryMiddleware(max_retries=2, backoff_factor=2.0, initial_delay=1.0),
        ModelFallbackMiddleware("openai:gpt-4o-mini"),
    ],
)
r = resilient_agent.invoke({"messages": "Say the word: fallback-worked"})
print(r["messages"][-1].content)
print("\nThe 'primary' model doesn't exist -- ModelFallbackMiddleware caught the failure and")
print("silently retried the whole call on gpt-4o-mini. The user never saw an error.")

# ToolRetryMiddleware -- shown as configuration (no flaky tool to demo against here):
print("\nToolRetryMiddleware(max_retries=2, tools=['search', 'fetch_url'], retry_on=(TimeoutError,))")
print("-- retries only the named tools, leaving everything else (e.g. send_email) untouched.")


Fallback-worked

The 'primary' model doesn't exist -- ModelFallbackMiddleware caught the failure and
silently retried the whole call on gpt-4o-mini. The user never saw an error.

ToolRetryMiddleware(max_retries=2, tools=['search', 'fetch_url'], retry_on=(TimeoutError,))
-- retries only the named tools, leaving everything else (e.g. send_email) untouched.


## 11. Structured output — when the answer needs to be a schema, not prose

A chatbot that has to hand its answer to another system (a CRM, a ticketing queue) can't
return free text. `response_format=ProviderStrategy(schema=...)` forces the final response
into a validated Pydantic model.


In [15]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy

class SupportTicket(BaseModel):
    category: str = Field(description="one of: billing, technical, account")
    urgency: str = Field(description="one of: low, medium, high")
    summary: str

structured_agent = create_agent(model=MODEL, tools=[], response_format=ProviderStrategy(schema=SupportTicket))

r = structured_agent.invoke({"messages": "My card was charged twice this month and I need this fixed urgently!"})
ticket = r["structured_response"]
print(type(ticket).__name__, ":", ticket)
print("\nThis is a real, validated SupportTicket object -- ready to hand straight to a ticketing API.")


SupportTicket : category='billing' urgency='high' summary='Double charge on my card this month'

This is a real, validated SupportTicket object -- ready to hand straight to a ticketing API.


## 12. Observability — see every model and tool call in production

Every agent above has been a black box: you only see the final answer. In production you
need to see *why* it answered that way — which tools ran, what the retrieved memories were,
where the latency went. **LangSmith tracing needs zero code changes** — set three
environment variables and every `create_agent` call in this notebook (or a real deployment)
is automatically traced.

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=my-chatbot   # optional, defaults to "default"
```

This course doesn't require a LangSmith account, so we don't force this to run — but this is
the actual, current, zero-code-change way production LangChain deployments get observability.
If `LANGSMITH_API_KEY` happens to be set, every call already made in this notebook was traced.


In [16]:
print("LANGSMITH_TRACING set:", bool(os.getenv("LANGSMITH_TRACING")))
print("LANGSMITH_API_KEY set:", bool(os.getenv("LANGSMITH_API_KEY")))
if not os.getenv("LANGSMITH_API_KEY"):
    print("\nNot configured in this environment -- nothing to trace to. The pattern above is what to add.")


LANGSMITH_TRACING set: False
LANGSMITH_API_KEY set: False

Not configured in this environment -- nothing to trace to. The pattern above is what to add.


## 13. Testing — prove it still works before every deploy

"It looked fine when I tried it" is not a test suite. The cheapest real test for an agent is
a **trajectory assertion**: not just "is the final text okay," but "did it call the right
tool, with the right arguments, in the right order." This needs no extra service — a plain
assert against `result["messages"]` catches regressions like "the agent stopped calling the
approval-gated tool" or "it now answers before checking memory."

(LangSmith also ships a proper trajectory-evaluator SDK for CI — `agentevals` +
`pytest.mark.langsmith` — for teams already using LangSmith; the assertion below is the
dependency-free version of the same idea.)


In [17]:
from langchain_core.messages import AIMessage, ToolMessage

def test_agent_calls_send_email_with_correct_recipient():
    result = hitl_agent.invoke(
        {"messages": "Send an email to alice@example.com, subject 'Test', body 'hello'"},
        {"configurable": {"thread_id": "test-thread-1"}},
    )
    # It should have paused for approval rather than sending immediately.
    assert "__interrupt__" in result, "expected the agent to pause for human approval"
    request = result["__interrupt__"][0].value["action_requests"][0]
    assert request["name"] == "send_email"
    assert request["args"]["recipient"] == "alice@example.com"
    print("PASS: send_email was gated behind approval with the correct recipient.")

test_agent_calls_send_email_with_correct_recipient()


PASS: send_email was gated behind approval with the correct recipient.


## 14. Putting it all together — one production-ready chatbot

Every middleware composes: they all just go in the same list, running in the order given.
This single agent has short-term memory, long-term memory, summarization, PII protection,
moderation, human approval on a sensitive tool, retries, and a model fallback — all at once.


In [18]:
production_agent = create_agent(
    model="openai:gpt-does-not-exist-9999",   # deliberately broken, to prove the fallback still saves it
    tools=[send_email],
    context_schema=Context,
    checkpointer=SqliteSaver(sqlite3.connect("production_chat.db", check_same_thread=False)),
    store=store,
    middleware=[
        # 1. Deterministic, cheap checks first
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        moderate_input,
        # 2. Personalization from long-term memory
        personalized_prompt,
        # 3. Cost control
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 2000), keep=("messages", 10)),
        # 4. Resilience
        ModelRetryMiddleware(max_retries=2),
        ModelFallbackMiddleware("openai:gpt-4o-mini"),
        # 5. Human approval on the one risky tool
        HumanInTheLoopMiddleware(interrupt_on={"send_email": {"allowed_decisions": ["approve", "reject"]}}),
    ],
)

config = {"configurable": {"thread_id": "prod-demo"}}
r = production_agent.invoke(
    {"messages": "Hi, what's my favorite programming language?"},
    config,
    context=Context(user_id="user-42"),
)
print("Personalized + resilient + moderated, all at once:")
print(" ", r["messages"][-1].content)


Personalized + resilient + moderated, all at once:
  Your favorite programming language is Rust.


## 15. If you see this online: old pattern → current equivalent

The open internet is still full of tutorials using the deprecated pattern. Recognize it, and
know what to reach for instead:

| Deprecated (still importable from `langchain_classic`, gone in 2.0) | Current equivalent |
|---|---|
| `ConversationBufferMemory` | `checkpointer=` on `create_agent` (`InMemorySaver`/`SqliteSaver`/`PostgresSaver`) |
| `ConversationSummaryMemory` / `ConversationSummaryBufferMemory` | `SummarizationMiddleware` |
| `ConversationBufferWindowMemory` / `ConversationTokenBufferMemory` | a `@before_model` trimming middleware |
| `RunnableWithMessageHistory` | `checkpointer=` + `thread_id` (same mechanism, no LCEL wrapper needed) |
| `AgentExecutor` + `create_tool_calling_agent` | `create_agent` |
| Hand-rolled "remember facts about the user" via a global dict | `store=` + namespaced `Store` + semantic search |

## Summary

| Concern | What handles it |
|---|---|
| One conversation, remembered | `checkpointer=` + `thread_id` |
| Survives a restart | `SqliteSaver` → `PostgresSaver`/`MongoDBSaver` |
| Cost doesn't grow unbounded | `SummarizationMiddleware` (keeps facts) or `@before_model` trimming (cheaper, lossy) |
| Remembers a user across sessions | `store=` + namespace + `@dynamic_prompt` |
| Real-time UI | `astream_events` |
| No PII/unsafe content leaks | `PIIMiddleware` + a moderation `@wrap_model_call` |
| Risky actions need a human | `HumanInTheLoopMiddleware` + `Command(resume=...)` |
| Provider outages don't take the product down | `ModelRetryMiddleware` + `ModelFallbackMiddleware` + `ToolRetryMiddleware` |
| Answers usable by other systems | `response_format=ProviderStrategy(schema=...)` |
| Visibility into what happened | LangSmith tracing (env vars only) |
| Confidence before every deploy | Trajectory assertions on `result["messages"]` |

### Next

This chatbot can hold a conversation, remember its user, protect itself, and recover from
failure -- but it still answers from what the model already knows. Wire the retrieval
pipeline from notebooks 05-08 into a tool this agent can call, and you have conversational
RAG: a chatbot that remembers *and* knows your documents.
